## Matrix solve

In fact, `expected_sojourn_time` produce the same result as R's solve. If we start from the matrices:

In [ ]:
import phasic
from phasic import Graph, StateIndexer, Property, with_ipv
phasic.configure(graph_cache=True, 
                 reward_compute_cache=True, 
                 parallel_elimination=True)

nr_samples = 8
indexer = StateIndexer(descendants=[
    Property('loc1', max_value=nr_samples),
    Property('loc2', max_value=nr_samples)
])

initial = [0] * indexer.state_length
initial[indexer.props_to_index(loc1=1, loc2=1)] = nr_samples

@with_ipv(initial)
def two_locus_arg_2param(state, indexer=None): # <- changed

    transitions = []
    if state.sum() <= 1: return transitions

    for i in range(indexer.state_length):
        if state[i] == 0: continue
        pi = indexer.index_to_props(i)

        for j in range(i, indexer.state_length):
            if state[j] == 0: continue
            pj = indexer.index_to_props(j)
            
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue 
            child = state.copy()
            child[i] -= 1
            child[j] -= 1
            loc1 = pi.descendants.loc1 + pj.descendants.loc1
            loc2 = pi.descendants.loc2 + pj.descendants.loc2
            if loc1 <= nr_samples and loc2 <= nr_samples:
                child[indexer.props_to_index(loc1=loc1, loc2=loc2)] += 1
                transitions.append([child, [state[i]*(state[j]-same)/(1+same), 0]])

        if state[i] > 0 and pi.descendants.loc1 > 0 and pi.descendants.loc2 > 0:
            child = state.copy()
            child[i] -= 1
            child[indexer.props_to_index(loc1=pi.descendants.loc1, loc2=0)] += 1
            child[indexer.props_to_index(loc1=0, loc2=pi.descendants.loc2)] += 1
            transitions.append([child, [0, 1]])

    return transitions

graph = Graph(two_locus_arg_2param, indexer=indexer) 
print(graph.vertices_length())
graph.clear_from_cache(graph_cache=False)

8407


{'graph_cache': 0, 'parameterized_reward_compute': 0}

`expected_sojourn_time` on a graph made from the matrices produces this:

In [ ]:
%%monitor
graph.update_weights([1.0, 1.0])
x = graph.expected_sojourn_time()

In [ ]:
%%time
x = graph.expected_sojourn_time()

CPU times: user 510 ms, sys: 27.8 ms, total: 538 ms
Wall time: 538 ms


In [ ]:
%%time
x = graph.expected_sojourn_time()

CPU times: user 511 ms, sys: 25.2 ms, total: 536 ms
Wall time: 536 ms


In [ ]:
from scipy.linalg import solve
mat = graph.as_matrices()
x = -solve(mat.sim.T, mat.ipv)
x

In [ ]:
%%time
x = -solve(SIM.T, IPV)

CPU times: user 537 ms, sys: 47.5 ms, total: 585 ms
Wall time: 183 ms
